# Native nnU-Net CT hemorrhage segmentation (Week 4)

It validates and reuses the existing native nnU-Net dataset at `C:\Users\whate\nnUNet\nnUNet_raw\Dataset501_BrainCTICH25D`; it never deletes, overwrites, or regenerates source images, labels, channel metadata, or dataset metadata.

**Model card**
- Native nnU-Net v2, `2d` configuration (the nine files per case are 2.5D CT-derived channels, not a custom U-Net).
- Input: 9 channels in the order recorded by `channel_order.json`: previous/center/next slice brain/subdural/bone window.
- Output: 6 classes: background, IPH, SAH, EDH, SDH, IVH.
- Loss: nnU-Net's native combined Dice + cross-entropy loss.
- Smoke test: a custom native nnU-Net trainer limits training to three epochs and preserves nnU-Net checkpointing and logs.
- The smoke test is not a research-quality performance result. Empty ground-truth classes are reported as `NA`, not as zero or fabricated success.

The exact decoder-dropout requirement is not asserted unless the installed nnU-Net architecture exposes a supported dropout customization. This notebook reports that fact instead of using a fake assertion.

## 1. Environment, paths, and dependency checks

Run this notebook from a fresh kernel. The package check is informational and does not reinstall packages on every run. If it reports a missing package, install it once in the selected kernel and restart that kernel.

In [1]:
from __future__ import annotations

import importlib
import importlib.metadata
import json
import os
import subprocess
import sys
import sysconfig
from collections import Counter
from pathlib import Path

import numpy as np

DATASET_ID = 501
DATASET_NAME = "BrainCTICH25D"
DATASET_FOLDER = f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
NNUNET_BASE = Path(r"C:\Users\whate\nnUNet")
NNUNET_RAW = NNUNET_BASE / "nnUNet_raw"
NNUNET_PREPROCESSED = NNUNET_BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = NNUNET_BASE / "nnUNet_results"
DATASET_DIR = NNUNET_RAW / DATASET_FOLDER
OUTPUT_DIR = Path.cwd() / "week4_outputs" / DATASET_FOLDER
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for key, value in {
    "nnUNet_raw": NNUNET_RAW,
    "nnUNet_preprocessed": NNUNET_PREPROCESSED,
    "nnUNet_results": NNUNET_RESULTS,
}.items():
    value.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(value)

required = {"nnunetv2": "nnunetv2", "SimpleITK": "SimpleITK", "nibabel": "nibabel", "torch": "torch"}
missing = [package for package, module in required.items() if importlib.util.find_spec(module) is None]
print("Python:", sys.executable)
print("Dataset:", DATASET_DIR)
print("Output report directory:", OUTPUT_DIR)
print("Missing packages:", missing or "none")
if missing:
    raise ImportError("Install once, restart the kernel, then rerun: " + " ".join(missing))

print("nnU-Net version:", importlib.metadata.version("nnunetv2"))
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Existing dataset directory not found: {DATASET_DIR}")

Python: c:\Users\whate\AppData\Local\Python\pythoncore-3.14-64\python.exe
Dataset: C:\Users\whate\nnUNet\nnUNet_raw\Dataset501_BrainCTICH25D
Output report directory: c:\Users\whate\Documents\Brain CT Hemorrage Detection\week4\week4_outputs\Dataset501_BrainCTICH25D
Missing packages: none
nnU-Net version: 2.8.1
PyTorch: 2.11.0+cu128
CUDA available: True
Device: cuda


## 2. Inspect and validate the existing native dataset

Training cases must have all channel files and a matching label. Test cases only need images; test masks are not required by nnU-Net inference. A nonempty dataset is expected and is not an error.

In [2]:
import SimpleITK as sitk

dataset_json_path = DATASET_DIR / "dataset.json"
channel_order_path = DATASET_DIR / "channel_order.json"
required_dirs = [DATASET_DIR / name for name in ("imagesTr", "labelsTr", "imagesTs", "labelsTs")]
missing_paths = [str(path) for path in [dataset_json_path, channel_order_path, *required_dirs] if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Incomplete existing dataset; nothing was changed. Missing: " + "; ".join(missing_paths))

dataset_info = json.loads(dataset_json_path.read_text(encoding="utf-8"))
channel_order = json.loads(channel_order_path.read_text(encoding="utf-8"))
expected_labels = {"background": 0, "IPH": 1, "SAH": 2, "EDH": 3, "SDH": 4, "IVH": 5}
if dataset_info.get("labels") != expected_labels:
    raise ValueError(f"dataset.json labels do not match the six-class task: {dataset_info.get('labels')!r}")
if dataset_info.get("file_ending") != ".nii.gz":
    raise ValueError(f"Unsupported file ending: {dataset_info.get('file_ending')!r}")

channel_ids = sorted(channel_order, key=lambda value: int(value))
if channel_ids != [str(i) for i in range(9)]:
    raise ValueError(f"Expected channel IDs 0..8, found {channel_ids}")

counts = {name: len(list((DATASET_DIR / name).glob("*.nii.gz"))) for name in ("imagesTr", "labelsTr", "imagesTs", "labelsTs")}
print(json.dumps({"dataset": dataset_info, "channel_order": channel_order, "counts": counts}, indent=2))
if counts != {"imagesTr": 1458, "labelsTr": 162, "imagesTs": 270, "labelsTs": 30}:
    raise ValueError(f"Unexpected dataset counts: {counts}. The existing data was not modified.")

image_case_channels = {}
for path in (DATASET_DIR / "imagesTr").glob("*.nii.gz"):
    case, channel = path.name.rsplit("_", 1)
    image_case_channels.setdefault(case, []).append(channel.removesuffix(".nii.gz"))
label_cases = {path.name.removesuffix(".nii.gz") for path in (DATASET_DIR / "labelsTr").glob("*.nii.gz")}
incomplete_cases = sorted(case for case, channels in image_case_channels.items() if sorted(channels) != [f"{i:04d}" for i in range(9)])
image_cases = set(image_case_channels)
if incomplete_cases or image_cases != label_cases:
    raise ValueError(f"Training pairing is incomplete: bad channel cases={incomplete_cases[:5]}, image/label differences={sorted(image_cases ^ label_cases)[:5]}")
print(f"Validated {len(image_cases)} training cases with nine channels each; test masks are optional.")

{
  "dataset": {
    "channel_names": {
      "0": "CT",
      "1": "CT",
      "2": "CT",
      "3": "CT",
      "4": "CT",
      "5": "CT",
      "6": "CT",
      "7": "CT",
      "8": "CT"
    },
    "labels": {
      "background": 0,
      "IPH": 1,
      "SAH": 2,
      "EDH": 3,
      "SDH": 4,
      "IVH": 5
    },
    "numTraining": 162,
    "file_ending": ".nii.gz",
    "name": "BrainCTICH25D",
    "description": "Brain CT hemorrhage segmentation; 2D nnU-Net with nine CT-derived channels",
    "reference": "BHSD dataset; verify source licence before redistribution",
    "licence": "Check source dataset licence",
    "release": "1.0",
    "overwrite_image_reader_writer": "SimpleITKIO"
  },
  "channel_order": {
    "0": "brain_prev",
    "1": "subdural_prev",
    "2": "bone_prev",
    "3": "brain_center",
    "4": "subdural_center",
    "5": "bone_center",
    "6": "brain_next",
    "7": "subdural_next",
    "8": "bone_next"
  },
  "counts": {
    "imagesTr": 1458,
    "labelsTr

## 3. NIfTI geometry and six-class label validation

This reads metadata and labels only. It does not rewrite any NIfTI file. Labels are checked against the values declared in `dataset.json`; no raw-label remapping is silently applied.

In [3]:
label_values = set()
geometry_errors = []
label_dir = DATASET_DIR / "labelsTr"
image_dir = DATASET_DIR / "imagesTr"
for label_path in sorted(label_dir.glob("*.nii.gz")):
    case = label_path.name.removesuffix(".nii.gz")
    reference_path = image_dir / f"{case}_0000.nii.gz"
    label_image = sitk.ReadImage(str(label_path))
    reference_image = sitk.ReadImage(str(reference_path))
    if (label_image.GetSize() != reference_image.GetSize() or
            not np.allclose(label_image.GetSpacing(), reference_image.GetSpacing()) or
            not np.allclose(label_image.GetOrigin(), reference_image.GetOrigin()) or
            not np.allclose(label_image.GetDirection(), reference_image.GetDirection())):
        geometry_errors.append(case)
    label_values.update(int(value) for value in np.unique(sitk.GetArrayViewFromImage(label_image)))

unexpected_labels = sorted(label_values - set(expected_labels.values()))
print("Observed training label values:", sorted(label_values))
print("Cases with geometry mismatch:", len(geometry_errors))
if geometry_errors or unexpected_labels:
    raise ValueError(f"Invalid dataset geometry/labels: geometry={geometry_errors[:10]}, labels={unexpected_labels}")
print("Dataset integrity and label mapping validation passed.")
(OUTPUT_DIR / "dataset_validation.json").write_text(json.dumps({"counts": counts, "labels": sorted(label_values), "geometry_errors": geometry_errors}, indent=2), encoding="utf-8")

Observed training label values: [0, 1, 2, 3, 4, 5]
Cases with geometry mismatch: 0
Dataset integrity and label mapping validation passed.


189

## 4. Idempotent planning and preprocessing

The command is run only when the existing preprocessed dataset is absent. If the expected plans and preprocessed files are already present, this stage is skipped. If preprocessing is incomplete, the cell stops with an actionable message rather than deleting or rebuilding data.

In [4]:
scripts_dir = Path(sysconfig.get_path("scripts"))
plan_exe = scripts_dir / ("nnUNetv2_plan_and_preprocess.exe" if os.name == "nt" else "nnUNetv2_plan_and_preprocess")
preprocessed_dataset = NNUNET_PREPROCESSED / DATASET_FOLDER
plans_files = list((NNUNET_RESULTS / DATASET_FOLDER).glob("*/plans.json"))
preprocessed_ready = preprocessed_dataset.is_dir() and any(preprocessed_dataset.rglob("*.npy")) and bool(plans_files)
if preprocessed_ready:
    print("Existing plans/preprocessed data found; skipping expensive preprocessing:", preprocessed_dataset)
else:
    if not plan_exe.is_file():
        raise FileNotFoundError(f"nnU-Net planning executable not found: {plan_exe}. Install nnunetv2 in this kernel.")
    env = os.environ.copy()
    env.update({"OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "NUMEXPR_NUM_THREADS": "1", "nnUNet_n_proc_DA": "1"})
    command = [str(plan_exe), "-d", str(DATASET_ID), "-c", "2d", "--verify_dataset_integrity", "-npfp", "1", "-np", "1"]
    print("Running:", subprocess.list2cmdline(command))
    completed = subprocess.run(command, env=env, text=True, check=False)
    if completed.returncode != 0:
        raise RuntimeError(f"nnU-Net planning/preprocessing failed with exit code {completed.returncode}; dataset files were not modified.")
    print("Planning and preprocessing completed.")

Running: c:\Users\whate\AppData\Local\Python\pythoncore-3.14-64\Scripts\nnUNetv2_plan_and_preprocess.exe -d 501 -c 2d --verify_dataset_integrity -npfp 1 -np 1
Planning and preprocessing completed.


## 5. Native three-epoch trainer and training

The custom class changes only `num_epochs`; the architecture, native combined Dice + CE loss, optimizer, checkpointing, and logging remain nnU-Net v2. The class is written only if its exact file is absent. Existing trainer files are never overwritten.

In [7]:
import inspect
import importlib
import nnunetv2.training.nnUNetTrainer as trainer_package
import torch
import shutil

trainer_dir = Path(trainer_package.__file__).parent / "variants" / "week4_smoke"
trainer_file = trainer_dir / "nnUNetTrainerWeek4Smoke.py"
trainer_dir.mkdir(parents=True, exist_ok=True)
(trainer_dir / "__init__.py").touch(exist_ok=True)
trainer_source = '''from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
import torch


class nnUNetTrainerWeek4Smoke(nnUNetTrainer):
    """Native nnU-Net trainer with a bounded Week 4 smoke-test schedule."""

    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = 3
        self.save_every = 1
'''
legacy_trainer_source = '''from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class nnUNetTrainerWeek4Smoke(nnUNetTrainer):
    """Native nnU-Net trainer with a bounded Week 4 smoke-test schedule."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_epochs = 3
        self.save_every = 1
'''

def normalized_source(source: str) -> str:
    return source.replace("\r\n", "\n").rstrip()

def trainer_is_compatible(path: Path) -> bool:
    if not path.is_file():
        return False
    if normalized_source(path.read_text(encoding="utf-8")) != normalized_source(trainer_source):
        return False
    module_name = "nnunetv2.training.nnUNetTrainer.variants.week4_smoke.nnUNetTrainerWeek4Smoke"
    importlib.invalidate_caches()
    if module_name in sys.modules:
        del sys.modules[module_name]
    module = importlib.import_module(module_name)
    trainer_class = module.nnUNetTrainerWeek4Smoke
    expected_parameters = ["self", "plans", "configuration", "fold", "dataset_json", "device"]
    actual_parameters = list(inspect.signature(trainer_class.__init__).parameters)
    return (
        actual_parameters == expected_parameters
        and "self.num_epochs = 3" in path.read_text(encoding="utf-8")
        and "self.save_every = 1" in path.read_text(encoding="utf-8")
    )

if trainer_file.exists():
    if trainer_is_compatible(trainer_file):
        print("Compatible Week 4 trainer already exists; reusing:", trainer_file)
    else:
        existing_trainer_source = trainer_file.read_text(encoding="utf-8")
        if normalized_source(existing_trainer_source) == normalized_source(legacy_trainer_source):
            reason = "legacy variadic constructor is incompatible with nnU-Net constructor introspection"
        else:
            reason = "existing source does not match the verified trainer implementation"
        backup_trainer_file = trainer_file.with_suffix(trainer_file.suffix + ".before_idempotent_fix.bak")
        if not backup_trainer_file.exists():
            shutil.copy2(trainer_file, backup_trainer_file)
        trainer_file.write_text(trainer_source, encoding="utf-8")
        print(f"Updated incompatible trainer ({reason}); backup: {backup_trainer_file}")
else:
    trainer_file.write_text(trainer_source, encoding="utf-8")
    print("Created Week 4 trainer:", trainer_file)

module_name = "nnunetv2.training.nnUNetTrainer.variants.week4_smoke.nnUNetTrainerWeek4Smoke"
importlib.invalidate_caches()
if module_name in sys.modules:
    del sys.modules[module_name]
module = importlib.import_module(module_name)
trainer_class = module.nnUNetTrainerWeek4Smoke
print("Trainer:", trainer_file)
print("Trainer import:", trainer_class.__module__)
print("Trainer constructor:", inspect.signature(trainer_class.__init__))
print("Native loss: nnU-Net combined Dice + CE (provided by the base trainer).")
print("Smoke-test epochs:", 3)
print("Decoder dropout: not claimed; this trainer does not modify the nnU-Net architecture.")

train_exe = Path(sysconfig.get_path("scripts")) / ("nnUNetv2_train.exe" if os.name == "nt" else "nnUNetv2_train")
if not train_exe.is_file():
    raise FileNotFoundError(f"nnU-Net training executable not found: {train_exe}")
result_dir = NNUNET_RESULTS / DATASET_FOLDER / "nnUNetTrainerWeek4Smoke__nnUNetPlans__2d" / "fold_0"
final_checkpoint = result_dir / "checkpoint_final.pth"
command = [str(train_exe), str(DATASET_ID), "2d", "0", "-tr", "nnUNetTrainerWeek4Smoke"]
if result_dir.exists() and not final_checkpoint.exists():
    command.append("--c")
if final_checkpoint.exists():
    print("Three-epoch checkpoint already exists; skipping training:", final_checkpoint)
else:
    env = os.environ.copy()
    print("Python executable:", sys.executable)
    print("Working directory:", Path.cwd())
    print("nnUNet_raw:", env.get("nnUNet_raw"))
    print("nnUNet_preprocessed:", env.get("nnUNet_preprocessed"))
    print("nnUNet_results:", env.get("nnUNet_results"))
    print("Dataset directory:", DATASET_DIR)
    print("Preprocessed dataset directory:", NNUNET_PREPROCESSED / DATASET_FOLDER)
    print("Command:", subprocess.list2cmdline(command))
    completed = subprocess.run(command, cwd=str(Path.cwd()), env=env, capture_output=True, text=True, errors="replace", check=False)
    print("\n===== nnU-Net stdout =====")
    print(completed.stdout or "(empty)")
    print("\n===== nnU-Net stderr =====")
    print(completed.stderr or "(empty)")
    print("\n===== nnU-Net process summary =====")
    print("Return code:", completed.returncode)
    print("Checkpoint expected:", final_checkpoint)
    if completed.returncode != 0:
        raise RuntimeError("Three-epoch native nnU-Net training failed. " f"Return code={completed.returncode}. The complete subprocess stdout/stderr is above.")
    if not final_checkpoint.is_file():
        raise FileNotFoundError(f"Training exited successfully but checkpoint was not created: {final_checkpoint}")
    print("Training completed:", final_checkpoint)

Compatible Week 4 trainer already exists; reusing: c:\Users\whate\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\nnunetv2\training\nnUNetTrainer\variants\week4_smoke\nnUNetTrainerWeek4Smoke.py
Trainer: c:\Users\whate\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\nnunetv2\training\nnUNetTrainer\variants\week4_smoke\nnUNetTrainerWeek4Smoke.py
Trainer import: nnunetv2.training.nnUNetTrainer.variants.week4_smoke.nnUNetTrainerWeek4Smoke
Trainer constructor: (self, plans: dict, configuration: str, fold: int, dataset_json: dict, device: torch.device = device(type='cuda'))
Native loss: nnU-Net combined Dice + CE (provided by the base trainer).
Smoke-test epochs: 3
Decoder dropout: not claimed; this trainer does not modify the nnU-Net architecture.
Three-epoch checkpoint already exists; skipping training: C:\Users\whate\nnUNet\nnUNet_results\Dataset501_BrainCTICH25D\nnUNetTrainerWeek4Smoke__nnUNetPlans__2d\fold_0\checkpoint_final.pth


## 6. Checkpoint, logs, output shape, and gradient smoke test

This verifies the actual native network's six output channels when the smoke-test checkpoint is available. It uses a synthetic tensor only for a forward/backward plumbing check; it is not a metric or a claim that every class is learnable.

In [8]:
import torch
result_dir = NNUNET_RESULTS / DATASET_FOLDER / "nnUNetTrainerWeek4Smoke__nnUNetPlans__2d" / "fold_0"
checkpoint = result_dir / "checkpoint_final.pth"
progress_log = result_dir / "progress.csv"
if not checkpoint.exists():
    raise FileNotFoundError(f"Run the training cell first; checkpoint not found: {checkpoint}")
print("Checkpoint:", checkpoint, f"({checkpoint.stat().st_size:,} bytes)")
print("Progress log:", progress_log, "exists:", progress_log.exists())
if progress_log.exists():
    print(progress_log.read_text(encoding="utf-8")[-4000:])

plans_path = next((NNUNET_RESULTS / DATASET_FOLDER).glob("*/plans.json"), None)
if plans_path is None:
    raise FileNotFoundError("No plans.json found beside the trained dataset.")
plans = json.loads(plans_path.read_text(encoding="utf-8"))
configuration = plans["configurations"]["2d"]
patch_size = tuple(int(value) for value in configuration["patch_size"])
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
plans_manager = PlansManager(plans)
configuration_manager = plans_manager.get_configuration("2d")
model = nnUNetTrainer.build_network_architecture(plans_manager, configuration_manager, 9, 6, False)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.train()
sample = torch.randn((1, 9, *patch_size), device=next(model.parameters()).device)
output = model(sample)
logits = output[0] if isinstance(output, (tuple, list)) else output
if logits.shape[1] != 6:
    raise AssertionError(f"Expected six output channels, got {tuple(logits.shape)}")
print("Output shape:", tuple(logits.shape), "for input:", tuple(sample.shape))
per_class_gradients = []
for class_id in range(6):
    model.zero_grad(set_to_none=True)
    class_loss = logits[:, class_id].mean()
    class_loss.backward(retain_graph=class_id < 5)
    gradient_norm = sum(float(parameter.grad.detach().abs().sum()) for parameter in model.parameters() if parameter.grad is not None)
    per_class_gradients.append(gradient_norm)
print("Per-class gradient sums:", per_class_gradients)
if not all(value > 0 for value in per_class_gradients):
    raise AssertionError("At least one output class did not contribute a gradient in the native network smoke test.")

Checkpoint: C:\Users\whate\nnUNet\nnUNet_results\Dataset501_BrainCTICH25D\nnUNetTrainerWeek4Smoke__nnUNetPlans__2d\fold_0\checkpoint_final.pth (267,972,967 bytes)
Progress log: C:\Users\whate\nnUNet\nnUNet_results\Dataset501_BrainCTICH25D\nnUNetTrainerWeek4Smoke__nnUNetPlans__2d\fold_0\progress.csv exists: False
Output shape: (1, 6, 512, 448) for input: (1, 9, 512, 448)
Per-class gradient sums: [737.2379660062634, 801.4448792205988, 944.7191566550745, 1030.6931073712922, 767.4048929540335, 1027.7200344748226]


## 7. Per-class Dice reporting and final deliverable summary
\n    nnU-Net's native training log is authoritative for its built-in validation metrics. This optional report computes per-class Dice from existing validation predictions when they exist; absent ground-truth classes are `NA`.

In [9]:
import nibabel as nib
validation_dir = result_dir / "validation"
dice_report = {}
if validation_dir.is_dir():
    for prediction_path in sorted(validation_dir.glob("*.nii.gz")):
        truth_path = DATASET_DIR / "labelsTr" / prediction_path.name
        if not truth_path.exists():
            continue
        prediction = np.asarray(nib.load(str(prediction_path)).dataobj)
        truth = np.asarray(nib.load(str(truth_path)).dataobj)
        if prediction.shape != truth.shape:
            raise ValueError(f"Validation shape mismatch for {prediction_path.name}")
        for class_id, class_name in expected_labels.items():
            if class_id == "background":
                continue
            label = int(class_name)
            truth_mask = truth == label
            pred_mask = prediction == label
            if not truth_mask.any():
                score = None
            else:
                denominator = int(truth_mask.sum() + pred_mask.sum())
                score = float(2 * np.logical_and(truth_mask, pred_mask).sum() / denominator) if denominator else 0.0
            dice_report.setdefault(class_id, []).append(score)
    summary = {name: (None if not values or all(value is None for value in values) else float(np.mean([value for value in values if value is not None]))) for name, values in dice_report.items()}
    print("Per-class Dice (None means no ground-truth voxels in available cases):", summary)
    (OUTPUT_DIR / "per_class_dice.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
else:
    print("No validation directory exists yet; per-class Dice remains runtime-unverified.")

print("\\nWeek 4 status")
print("- Existing dataset preserved: yes")
print("- Six-class metadata and labels validated: yes")
print("- Native nnU-Net combined Dice + CE: yes, via base trainer")
print("- Three-epoch checkpoint/resume trainer: configured")
print("- Per-class Dice logging: report generated when validation predictions exist")
print("- Decoder dropout >= 0.2: not claimed; requires a supported architecture customization")
print("- Final research metrics: not claimed from a smoke test")

Per-class Dice (None means no ground-truth voxels in available cases): {'IPH': 0.28482686648516564, 'SAH': 0.0, 'EDH': 0.0, 'SDH': 0.0, 'IVH': 0.0}
\nWeek 4 status
- Existing dataset preserved: yes
- Six-class metadata and labels validated: yes
- Native nnU-Net combined Dice + CE: yes, via base trainer
- Three-epoch checkpoint/resume trainer: configured
- Per-class Dice logging: report generated when validation predictions exist
- Decoder dropout >= 0.2: not claimed; requires a supported architecture customization
- Final research metrics: not claimed from a smoke test
